In [ ]:
from concurrent.futures import ThreadPoolExecutor
import threading
import queue
import numpy as np

In [4]:
class FlowGraph:
    """DAG of FunctionNodes with typed input/output.

    Each node's func: _out = func(_in).
    Output of one node becomes input of the next.
    Last node (no successors) returns None and handles
    its own output internally.

    Features:
      - Per-node concurrency limits
      - Data locality: same thread runs successor
      - Pending queue when node is at capacity

    In C++: maps to tbb::flow::graph with make_edge.
    """

    def __init__(self, num_workers=2):
        self.pool = ThreadPoolExecutor(max_workers=num_workers)

    def add_edge(self, src, dst):
        """Connect src's output to dst's input."""
        src.successors.append(dst)

    def try_put(self, node, _in):
        """Submit work to a node. 

        In C++: maps to input_node.try_put() or direct
        function_node.try_put().
        """
        self._try_claim_node(node, _in)

    def _try_claim_node(self, node, _in):
        """Try to claim capacity on a node. If full, queue for later."""
        with node._lock:
            if node._active < node.concurrency:
                node._active += 1
                self.pool.submit(self._execute_node, node, _in)
            else:
                node._pending.put(_in)

    def _execute_node(self, node, _in):
        """Runs on a POOL THREAD. Execute the node, then handle continuation.

        After the node finishes, try to continue to the successor
        directly on this thread (data locality). If the successor
        is full, queue the context and return to the pool.
        """
        # run the node's function
        _out = node.stage(_in)

        # last node: call done_callback instead of dropping result
        if not node.successors:
            self._release_node(node)
            if node._done_callback and _out is not None:
                node._done_callback(_out)
            return

        #  try to claim successor for data locality
        successor_claimed = False
        succ = node.successors[0]
        with succ._lock:
            if succ._active < succ.concurrency:
                succ._active += 1
                successor_claimed = True

        # release current node, pop pending if any
        # submit it to the pool (a different thread will pick it up).
        self._release_node(node)

        if successor_claimed:
            # DATA LOCALITY: run successor on THIS thread.
            # no pool submission, no queue. data stays in cache.
            self._execute_node(node.successors[0], _out)
        elif node.successors:
            # successor was full, queue for later.
            # when successor finishes its current work, _release_node
            # will pick this up from pending.
            self._try_claim_node(node.successors[0], _out)

    def _release_node(self, node):
        """Release one unit of capacity. If pending work exists, submit it."""
        with node._lock:
            node._active -= 1
            if not node._pending.empty() and node._active < node.concurrency:
                _in = node._pending.get()
                node._active += 1
                self.pool.submit(self._execute_node, node, _in)

    def shutdown(self):
        self.pool.shutdown(wait=False)

In [5]:
class FunctionNode:
    """A processing stage in the flow graph.

    Wraps a callable: _out = func(_in).
    The last node in a chain returns None — it handles
    its own side effects (print, store, send message, etc.)

    In C++: maps to tbb::flow::function_node<In, Out>.
    """

    def __init__(self, name, stage, concurrency=1, done_callback=None):
        self.name = name
        self.stage = stage
        self.func = getattr(stage, 'func', None)
        self.concurrency = concurrency
        self.successors = []
        self._active = 0
        self._lock = threading.Lock()
        self._pending = queue.Queue()
        self._done_callback = done_callback # callback for last node to handle output

    def __repr__(self):
        return f'FunctionNode({self.name}, concurrency={self.concurrency})'

In [6]:
class SlotPool:
    """Pre-allocated working memory for pipeline stages.

    Avoids per-message allocation. N slots bound memory and provide
    backpressure: acquire() returns None when all slots are busy,
    causing the actor to queue incoming messages until a slot is released.
    """
    
    def __init__(self, n_slots, slot_factory):
        self.slots = [slot_factory() for _ in range(n_slots)]
        self._free = list(range(n_slots))

    def acquire(self):
        if self._free:
            return self._free.pop(0)
        return None

    def release(self, slot_idx):
        self.slots[slot_idx].reset()
        self._free.append(slot_idx)

    def make_stage(self, func):
        """Create pipeline stage closure. Attaches pure_func for config access."""
        pool = self
        def stage(token):
            slot = pool.slots[token.slot]
            func(slot)
            return token
        stage.func = func 
        return stage

In [26]:
class ResizableSlotPool(SlotPool):
    """Safe resize: free slots immediately, in-flight on release."""

    def __init__(self, n_slots, slot_factory):
        super().__init__(n_slots, slot_factory)
        self._slot_size = len(self.slots[0].data)
        self._pending_resize = set()

    def set_size(self, new_size):
        self._slot_size = new_size
        for i in self._free:
            self.slots[i].data = np.empty(new_size, dtype=complex)
        all_slots = set(range(len(self.slots)))
        self._pending_resize = all_slots - set(self._free)

    def release(self, slot_idx):
        self.slots[slot_idx].reset()
        if slot_idx in self._pending_resize:
            self.slots[slot_idx].data = np.empty(self._slot_size, dtype=complex)
            self._pending_resize.discard(slot_idx)
        self._free.append(slot_idx)

#### Test
1. serial concurrency

In [7]:
import time

In [8]:
timeline = []

def slow_node(x):
    timeline.append(('start', x, threading.current_thread().name))
    time.sleep(0.05)
    timeline.append(('end', x, threading.current_thread().name))
    return x

g = FlowGraph(num_workers=4)
node = FunctionNode('slow', slow_node, concurrency=1)

g.try_put(node, 'A')
g.try_put(node, 'B') # second one should be queued

time.sleep(0.15)

# A must finish before B starts
assert timeline[1][:2] == ('end', 'A')
assert timeline[2][:2] == ('start', 'B')

2. parallel concurrency

In [9]:
timeline = []

g = FlowGraph(num_workers=4)
node = FunctionNode('slow', slow_node, concurrency=2)

g.try_put(node, 'A')
g.try_put(node, 'B') # second one should be queued

time.sleep(0.15)

# B will start before A ends
assert timeline[2][:2] == ('end', 'A')
assert timeline[1][:2] == ('start', 'B')

3. data locality + serial concurrency

In [10]:
threads_used = []

def node_a(x):
    threads_used.append(('node 1', threading.current_thread().name))
    time.sleep(0.05)
    return x + 1

def node_b(x):
    threads_used.append(('node 2', threading.current_thread().name))

g = FlowGraph(num_workers=4)
a = FunctionNode('node 1', node_a, concurrency=1)
b = FunctionNode('node 2', node_b, concurrency=1)
g.add_edge(a, b)

g.try_put(a, 10)
time.sleep(0.15)

assert threads_used[0][1] == threads_used[1][1]

4. data locality + parallel concurrency

In [11]:
threads_used = []

def node_a(x):
    threads_used.append(('node 1', x, threading.current_thread().name))
    time.sleep(0.05)
    return x 

def node_b(x):
    threads_used.append(('node 2', x, threading.current_thread().name))

g = FlowGraph(num_workers=4)
a = FunctionNode('node 1', node_a, concurrency=2)
b = FunctionNode('node 2', node_b, concurrency=2)
g.add_edge(a, b)

g.try_put(a, 'A')
g.try_put(a, 'B')
time.sleep(0.15)

assert threads_used[0][1:] == threads_used[2][1:] 
assert threads_used[1][1:] == threads_used[3][1:] 

5. parallel, later finishes first

In [12]:
threads_used = []

def node_a(x):
    threads_used.append(('node 1', x, threading.current_thread().name))
    if x == 'A':
        time.sleep(0.05)
    else:
        time.sleep(0.02)
    return x 

def node_b(x):
    threads_used.append(('node 2', x, threading.current_thread().name))

g = FlowGraph(num_workers=4)
a = FunctionNode('node 1', node_a, concurrency=2)
b = FunctionNode('node 2', node_b, concurrency=1)
g.add_edge(a, b)

g.try_put(a, 'A')
g.try_put(a, 'B')
time.sleep(0.15)

assert threads_used[0][1:] == threads_used[3][1:] 
assert threads_used[1][1:] == threads_used[2][1:] 

6. parallel faster than serial

In [13]:
from threading import Event

def slow_node(x):
    time.sleep(0.02)
    return x

# ---- serial ----
done = Event()
results = []

def sink(x):
    results.append(x)
    if len(results) >= 2:
        done.set()

g = FlowGraph(num_workers=4)
s = FunctionNode('slow', slow_node, concurrency=1)
k = FunctionNode('sink', sink, concurrency=1)
g.add_edge(s, k)

start = time.perf_counter()
g.try_put(s, 0)
g.try_put(s, 1)
done.wait()
serial_time = time.perf_counter() - start

# ---- parallel ----
done = Event()
results = []

s = FunctionNode('slow', slow_node, concurrency=2)
k = FunctionNode('sink', sink, concurrency=1)
g.add_edge(s, k)

start = time.perf_counter()
g.try_put(s, 0)
g.try_put(s, 1)
done.wait()
parallel_time = time.perf_counter(  ) - start

# ---- result ----
print(f"Serial time:   {serial_time:.3f} s")
print(f"Parallel time: {parallel_time:.3f} s")
print(f"Speedup:       {serial_time / parallel_time:.2f}x")

assert parallel_time < serial_time

Serial time:   0.041 s
Parallel time: 0.021 s
Speedup:       1.99x


7. slotpool: acquire and release

In [18]:
class TestSlot:
    def __init__(self):
        self.data = 0
    def reset(self):
        self.data = 0

pool = SlotPool(3, lambda: TestSlot())

s0 = pool.acquire()
s1 = pool.acquire()
s2 = pool.acquire()
s3 = pool.acquire()

assert s0 == 0
assert s1 == 1
assert s2 == 2
assert s3 is None            # all busy

pool.release(1)
s4 = pool.acquire()
assert s4 == 1               # slot 1 reused

8. slotpool: make stage

In [21]:
from dataclasses import dataclass

In [23]:
@dataclass
class Token:
    slot: int
    tag: dict

def double(slot):
    slot.data *= 2

pool = SlotPool(2, lambda: TestSlot())
stage = pool.make_stage(double)

pool.slots[0].data = 5
result = stage(Token(slot=0, tag={}))

assert pool.slots[0].data == 10   # function modified slot
assert result.slot == 0           # token passed through
assert stage.func is double       # pure function attached

9. resizableslotpool: resize slots

In [24]:
import numpy as np 

In [27]:
class ResizableTestSlot:
    def __init__(self, size):
        self.data = np.empty(size, dtype=complex)
        self.tag = None
    def reset(self):
        self.tag = None

# ---- Test 1: set_size resizes free slots immediately ----
pool = ResizableSlotPool(4, lambda: ResizableTestSlot(100))
assert len(pool.slots[0].data) == 100

pool.set_size(50)
# all slots are free → all resized immediately
for s in pool.slots:
    assert len(s.data) == 50
assert len(pool._pending_resize) == 0

10. resizableslotpool: in-flight slots resized on release

In [28]:
pool = ResizableSlotPool(4, lambda: ResizableTestSlot(100))

s0 = pool.acquire()   # slot 0 in-flight
s1 = pool.acquire()   # slot 1 in-flight
# slots 2, 3 are free

pool.set_size(50)

# free slots (2, 3) resized immediately
assert len(pool.slots[2].data) == 50
assert len(pool.slots[3].data) == 50

# in-flight slots (0, 1) still old size
assert len(pool.slots[0].data) == 100
assert len(pool.slots[1].data) == 100
assert pool._pending_resize == {0, 1}

# release slot 0 → resized now
pool.release(0)
assert len(pool.slots[0].data) == 50
assert pool._pending_resize == {1}

# release slot 1 → resized now
pool.release(1)
assert len(pool.slots[1].data) == 50
assert len(pool._pending_resize) == 0